# R4 Resume: Inference from Pre-trained Adapters (Both Seeds)

**Purpose:** Resume from crashed R4 runs. All 4 adapters are already trained and saved.
This notebook skips all training and runs **inference-only** for both seeds (42 & 123).

For each seed x variant (4 runs total):
1. Load pre-trained adapter
2. **Free-text inference** (saves raw text outputs for A3)
3. **Constrained decoding** (teacher-forced label scoring, zero parse failures)

**Crash recovery:** Each inference run saves a partial checkpoint to Drive.
Re-running the main cell auto-skips completed variants.

**Pre-requisite:** Upload adapters to `Drive/MyDrive/ViTHSD/adapters/`:
- `stage1_seed42/`, `stage1_seed123/` (Vanilla Stage 1)
- `rsqwen_seed42/`, `rsqwen_seed123/` (RSQwen Stage 2)

**Output per seed:**
- `results_constrained_seed{N}.json` - free-text + constrained metrics
- `raw_outputs_seed{N}.json` - raw model text outputs (for A3 re-parsing)
- `label_scores_seed{N}.json` - per-sample per-label log-prob scores
- `a3_postprocess_ablation_seed{N}.json` - prefer_hate vs prefer_offensive

## Step 1: Install Dependencies

In [1]:
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.0
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy tqdm openpyxl
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes
!pip install -q peft

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
warnings.filterwarnings('ignore')
print('Dependencies installed')

Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-monitoring 2.30.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
google-cloud-bigquery-connection 1.21.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
google-cloud-translate 3.26.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
googleapis-common-protos 1.74.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 3.20.0 which is incompatible.
wandb 0.25.1 requires protobuf!=5.28.0,!=5.29.0,<7,>4.21.0

## Step 2: Mount Google Drive & Setup Paths

Make sure `ViTHSD/src/` and `ViTHSD/adapters/` are up-to-date on Drive before running.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, sys, os
from pathlib import Path

# === CONFIGURE THIS ===
DRIVE_BASE = Path('/content/drive/MyDrive/ViTHSD')
# ======================

WORK_DIR = Path('/content/vithsd')
WORK_DIR.mkdir(exist_ok=True)

# Copy source files to working directory
for f in ['config.py', 'data_preparation.py', 'evaluation.py', 'models.py']:
    src = DRIVE_BASE / 'src' / f
    dst = WORK_DIR / f
    if src.exists():
        shutil.copy2(src, dst)
        print(f'  Copied {f}')
    else:
        raise FileNotFoundError(f'Missing: {src}')

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))

OUTPUT_DIR = DRIVE_BASE / 'outputs' / 'R4_constrained'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADAPTER_DIR = DRIVE_BASE / 'adapters'
assert ADAPTER_DIR.exists(), f'Missing adapter dir: {ADAPTER_DIR}'

# Verify all 4 adapters exist
for name in ['stage1_seed42', 'stage1_seed123', 'rsqwen_seed42', 'rsqwen_seed123']:
    p = ADAPTER_DIR / name / 'lora_adapter' / 'adapter_config.json'
    assert p.exists(), f'Missing adapter: {p}'
    print(f'  Found adapter: {name}')

print(f'\nWorking dir: {WORK_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Adapter dir: {ADAPTER_DIR}')

Mounted at /content/drive
  Copied config.py
  Copied data_preparation.py
  Copied evaluation.py
  Copied models.py
  Found adapter: stage1_seed42
  Found adapter: stage1_seed123
  Found adapter: rsqwen_seed42
  Found adapter: rsqwen_seed123

Working dir: /content/vithsd
Output dir: /content/drive/MyDrive/ViTHSD/outputs/R4_constrained
Adapter dir: /content/drive/MyDrive/ViTHSD/adapters


## Step 3: Patch Config & GPU Detection

In [4]:
import config
config.DATA_DIR = DRIVE_BASE / 'data'
config.OUTPUT_DIR = OUTPUT_DIR
config.BASE_DIR = WORK_DIR

for f in ['train.xlsx', 'test.xlsx']:
    p = config.DATA_DIR / f
    assert p.exists(), f'Missing data file: {p}'
    print(f'  Found: {f}')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > GPU')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

BATCH_SIZE = 8 if gpu_mem >= 35 else (4 if gpu_mem >= 14 else 2)
print(f'Batch size: {BATCH_SIZE}')
print('Config patched for Colab')

  Found: train.xlsx
  Found: test.xlsx

PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
Batch size: 8
Config patched for Colab


## Step 4: Load Data

In [5]:
import json
from config import FINAL_LABELS
from data_preparation import load_dataset_A
from evaluation import Evaluator
from models import create_model

train_texts, train_labels = load_dataset_A('train', data_dir=config.DATA_DIR)
test_texts, test_labels = load_dataset_A('test', data_dir=config.DATA_DIR)

print(f'Train: {len(train_texts)} | Test: {len(test_texts)}')
print(f'Labels: {FINAL_LABELS}')

[ViHSDLoader] Loaded 7000 samples from train
[ViHSDLoader] Loaded 1800 samples from test
Train: 7000 | Test: 1800
Labels: ['normal', 'individuals#offensive', 'individuals#hate', 'groups#offensive', 'groups#hate', 'religion#offensive', 'religion#hate', 'race#offensive', 'race#hate', 'politics#offensive', 'politics#hate']


## Step 5: Helper Functions

In [6]:
import random
import re
import numpy as np
import torch
import gc
import time
from sklearn.metrics import f1_score
from models import resolve_labels_list, convert_to_binary


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'  Seed set to {seed}')


def cleanup_model(model):
    if hasattr(model, 'model') and model.model is not None:
        del model.model
    if hasattr(model, 'tokenizer') and model.tokenizer is not None:
        del model.tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('  GPU memory freed')


def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def compute_parse_stats(raw_texts, parsed_labels, model_name):
    """Analyze parse failures in free-text outputs."""
    n = len(raw_texts)
    failures = 0
    for raw, labels in zip(raw_texts, parsed_labels):
        if labels == ['normal']:
            raw_lower = raw.lower().replace('<|im_end|>', '').strip()
            has_valid = any(l.lower() in raw_lower for l in FINAL_LABELS)
            if not has_valid:
                failures += 1
    return {'model': model_name, 'total': n, 'failures': failures, 'rate': failures / n}


def parse_raw_output(raw_text, prefer_hate=True):
    """Re-parse raw model text output with configurable conflict resolution."""
    output = raw_text.replace('<|im_end|>', '').strip()
    labels_match = re.search(r'labels?\s*(.+?)$', output, re.IGNORECASE | re.DOTALL)
    if labels_match:
        output = labels_match.group(1).strip()

    output_lower = output.lower()
    output_clean = output_lower.replace(';', ',').replace(' and ', ',')
    parts = [p.strip() for p in re.split(r'[,\n]', output_clean) if p.strip()]

    labels = []
    for part in parts:
        for valid_label in FINAL_LABELS:
            if valid_label.lower() == part or valid_label.lower() in part:
                if valid_label not in labels:
                    labels.append(valid_label)
                break

    if not labels:
        labels = ['normal']

    return resolve_labels_list(labels, prefer_hate=prefer_hate)


def bootstrap_test(y_true_bin, y_pred_a_bin, y_pred_b_bin, n_iter=10000, seed=42):
    """Paired bootstrap significance test on F1-Micro."""
    rng = np.random.RandomState(seed)
    n = len(y_true_bin)

    score_a = f1_score(y_true_bin, y_pred_a_bin, average='micro')
    score_b = f1_score(y_true_bin, y_pred_b_bin, average='micro')
    observed_diff = score_a - score_b

    count = 0
    for _ in range(n_iter):
        idx = rng.randint(0, n, size=n)
        sa = f1_score(y_true_bin[idx], y_pred_a_bin[idx], average='micro')
        sb = f1_score(y_true_bin[idx], y_pred_b_bin[idx], average='micro')
        if (sa - sb) <= 0:
            count += 1

    p_value = count / n_iter
    return {
        'score_a': float(score_a), 'score_b': float(score_b),
        'diff': float(observed_diff), 'p_value': float(p_value),
        'significant_0.05': p_value < 0.05, 'significant_0.01': p_value < 0.01
    }


print('Helper functions defined')

Helper functions defined


## Step 6: Initialize Result Containers (Both Seeds)

In [7]:
SEEDS = [42, 123]

# Per-seed containers
all_results = {}     # seed -> {'freetext': {}, 'constrained': {}}
all_raw_outputs = {} # seed -> {key: [raw_texts]}
all_label_scores = {}# seed -> {key: scores}

for s in SEEDS:
    all_results[s] = {'freetext': {}, 'constrained': {}}
    all_raw_outputs[s] = {}
    all_label_scores[s] = {}

print(f'Configured multi-seed run: seeds={SEEDS}')

Configured multi-seed run: seeds=[42, 123]


## Step 7: Main Inference Loop (Crash-Recoverable)

For each seed x variant (4 runs total):
1. Check if partial checkpoint exists -> skip if already done
2. Load base model + pre-trained adapter (no training)
3. Free-text predict -> constrained predict -> evaluate
4. Save partial checkpoint to Drive -> cleanup GPU

**If Colab crashes, just re-run this cell.** Completed variants are auto-skipped.

In [8]:
evaluator = Evaluator()

VARIANTS = [
    ('vanilla', 'stage1_seed{}'),
    ('rsqwen', 'rsqwen_seed{}'),
]

total_t0 = time.time()

for seed in SEEDS:
    print(f'\n{"="*70}')
    print(f'SEED {seed}')
    print(f'{"="*70}')
    seed_t0 = time.time()

    for variant_name, adapter_template in VARIANTS:
        key = f'{variant_name}_{seed}'
        adapter_subdir = adapter_template.format(seed)
        adapter_path = str(ADAPTER_DIR / adapter_subdir)

        # --- Crash recovery: check if this variant is already done ---
        partial_path = OUTPUT_DIR / f'{key}_checkpoint.json'
        if partial_path.exists():
            print(f'\n  --- {variant_name.upper()} seed={seed} --- SKIPPED (checkpoint exists)')
            with open(partial_path, 'r') as f:
                checkpoint = json.load(f)
            all_results[seed]['freetext'][key] = checkpoint['res_ft']
            all_results[seed]['constrained'][key] = checkpoint['res_cd']
            all_raw_outputs[seed][key] = checkpoint['raw_texts']
            all_label_scores[seed][key] = checkpoint['scores']
            print(f'    Loaded: FT F1-Micro={checkpoint["res_ft"]["f1_micro"]:.4f}, CD F1-Micro={checkpoint["res_cd"]["f1_micro"]:.4f}')
            continue

        print(f'\n  --- {variant_name.upper()} seed={seed} ---')
        t0 = time.time()

        set_seed(seed)
        model = create_model(
            'qwen', dataset_type='A',
            batch_size=BATCH_SIZE, num_epochs=2,
            learning_rate=2e-4, lora_r=8, lora_alpha=16
        )
        model.load_adapter(adapter_path)

        # Free-text inference
        print('  Free-text predicting...')
        pred_ft, raw_texts = model.predict(test_texts)
        res_ft = evaluator.evaluate(test_labels, pred_ft, f'{variant_name}_FreeText', f'seed_{seed}')

        # Constrained decoding inference
        print('  Constrained predicting...')
        pred_cd, scores = model.predict_constrained(test_texts)
        res_cd = evaluator.evaluate(test_labels, pred_cd, f'{variant_name}_Constrained', f'seed_{seed}')

        elapsed = (time.time() - t0) / 60
        print(f'  {variant_name.upper()} done in {elapsed:.1f} min')
        print(f'    FreeText:    F1-Micro={res_ft["f1_micro"]:.4f} F1-Macro={res_ft["f1_macro"]:.4f}')
        print(f'    Constrained: F1-Micro={res_cd["f1_micro"]:.4f} F1-Macro={res_cd["f1_macro"]:.4f}')

        # Store in memory
        res_ft_s = convert_to_serializable(res_ft)
        res_cd_s = convert_to_serializable(res_cd)
        scores_s = convert_to_serializable(scores)

        all_results[seed]['freetext'][key] = res_ft_s
        all_results[seed]['constrained'][key] = res_cd_s
        all_raw_outputs[seed][key] = raw_texts
        all_label_scores[seed][key] = scores_s

        # Save checkpoint to Drive (crash recovery)
        checkpoint = {
            'res_ft': res_ft_s,
            'res_cd': res_cd_s,
            'raw_texts': raw_texts,
            'scores': scores_s,
        }
        with open(partial_path, 'w') as f:
            json.dump(checkpoint, f, indent=2, ensure_ascii=False)
        print(f'  Checkpoint saved: {partial_path.name}')

        cleanup_model(model)

    seed_elapsed = (time.time() - seed_t0) / 60
    print(f'\n  Seed {seed} complete in {seed_elapsed:.1f} min')

total_elapsed = (time.time() - total_t0) / 60
print(f'\n{"="*70}')
print(f'ALL INFERENCE COMPLETE in {total_elapsed:.1f} min')
print(f'{"="*70}')


SEED 42

  --- VANILLA seed=42 ---
  Seed set to 42


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827
  Adapter loaded from /content/drive/MyDrive/ViTHSD/adapters/rsqwen_seed123
  Free-text predicting...


Predicting: 100%|██████████| 1800/1800 [37:57<00:00,  1.27s/it]


  Constrained predicting...


Constrained predicting: 100%|██████████| 1800/1800 [1:06:48<00:00,  2.23s/it]


  RSQWEN done in 104.9 min
    FreeText:    F1-Micro=0.5587 F1-Macro=0.3661
    Constrained: F1-Micro=0.2059 F1-Macro=0.0934
  Checkpoint saved: rsqwen_123_checkpoint.json
  GPU memory freed

  Seed 123 complete in 201.2 min

ALL INFERENCE COMPLETE in 392.9 min


## Step 8: Aggregate Results & Comparison

In [9]:
print(f'{"="*80}')
print(f'{"MULTI-SEED: CONSTRAINED vs FREE-TEXT":^80}')
print(f'{"="*80}')
print(f'{"":>30} {"F1-Micro":>12} {"F1-Macro":>12}')
print(f'{"-"*60}')

for seed in SEEDS:
    print(f'\n  Seed {seed}:')
    for model_prefix in ['vanilla', 'rsqwen']:
        key = f'{model_prefix}_{seed}'
        for mode in ['freetext', 'constrained']:
            if key in all_results[seed][mode]:
                r = all_results[seed][mode][key]
                tag = f'{model_prefix}_{mode}'
                print(f'    {tag:>26} {r["f1_micro"]*100:10.2f}% {r["f1_macro"]*100:10.2f}%')

print(f'\n{"-"*60}')
print('Constrained decoding delta (RSQwen - Vanilla):')
for seed in SEEDS:
    key_rs = f'rsqwen_{seed}'
    key_van = f'vanilla_{seed}'
    if key_rs in all_results[seed]['constrained'] and key_van in all_results[seed]['constrained']:
        diff = all_results[seed]['constrained'][key_rs]['f1_micro'] - all_results[seed]['constrained'][key_van]['f1_micro']
        winner = 'RSQwen' if diff > 0 else 'Vanilla'
        print(f'  Seed {seed}: {diff*100:+.2f}% F1-Micro -> {winner} wins')

                      MULTI-SEED: CONSTRAINED vs FREE-TEXT                      
                                   F1-Micro     F1-Macro
------------------------------------------------------------

  Seed 42:
              vanilla_freetext      60.00%      32.26%
           vanilla_constrained      16.08%       7.36%
               rsqwen_freetext      57.44%      35.37%
            rsqwen_constrained      15.25%       7.08%

  Seed 123:
              vanilla_freetext      59.66%      33.32%
           vanilla_constrained      26.95%      12.57%
               rsqwen_freetext      55.87%      36.61%
            rsqwen_constrained      20.59%       9.34%

------------------------------------------------------------
Constrained decoding delta (RSQwen - Vanilla):
  Seed 42: -0.84% F1-Micro -> Vanilla wins
  Seed 123: -6.36% F1-Micro -> Vanilla wins


## Step 9: Parse Failure Analysis (A3 data)

In [10]:
print(f'{"PARSE FAILURE RATES (FREE-TEXT)":^60}')
print(f'{"-"*60}')

for seed in SEEDS:
    for key, raw_list in all_raw_outputs[seed].items():
        stats = compute_parse_stats(raw_list, [], key)
        print(f'  {key}: {stats["failures"]}/{stats["total"]} = {stats["rate"]*100:.1f}% failures')

print(f'\n  Constrained decoding: 0% parse failures by construction')

              PARSE FAILURE RATES (FREE-TEXT)               
------------------------------------------------------------
  vanilla_42: 0/1800 = 0.0% failures
  rsqwen_42: 0/1800 = 0.0% failures
  vanilla_123: 0/1800 = 0.0% failures
  rsqwen_123: 0/1800 = 0.0% failures

  Constrained decoding: 0% parse failures by construction


## Step 10: A3 - Re-parse Raw Outputs with prefer_hate Ablation

In [11]:
all_a3_results = {}

for seed in SEEDS:
    all_a3_results[seed] = {}
    for key, raw_list in all_raw_outputs[seed].items():
        for prefer_hate in [True, False]:
            suffix = 'prefer_hate' if prefer_hate else 'prefer_offensive'
            reparsed = [parse_raw_output(r, prefer_hate=prefer_hate) for r in raw_list]
            res = evaluator.evaluate(test_labels, reparsed, f'{key}_{suffix}', 'a3')
            a3_key = f'{key}_{suffix}'
            all_a3_results[seed][a3_key] = convert_to_serializable(res)

print(f'{"A3: POST-PROCESSING RULE ABLATION":^70}')
print(f'{"="*70}')
print(f'{"":>40} {"F1-Micro":>12} {"F1-Macro":>12}')
print(f'{"-"*65}')

for seed in SEEDS:
    print(f'\n  Seed {seed}:')
    for key in sorted(all_a3_results[seed].keys()):
        r = all_a3_results[seed][key]
        print(f'    {key:>36} {r["f1_micro"]*100:10.2f}% {r["f1_macro"]*100:10.2f}%')

print(f'\nIf prefer_hate vs prefer_offensive shows minimal difference,')
print(f'the conflict resolution rule has negligible impact.')

                  A3: POST-PROCESSING RULE ABLATION                   
                                             F1-Micro     F1-Macro
-----------------------------------------------------------------

  Seed 42:
                   rsqwen_42_prefer_hate      57.44%      35.37%
              rsqwen_42_prefer_offensive      57.44%      35.37%
                  vanilla_42_prefer_hate      60.00%      32.26%
             vanilla_42_prefer_offensive      60.00%      32.26%

  Seed 123:
                  rsqwen_123_prefer_hate      55.87%      36.61%
             rsqwen_123_prefer_offensive      55.87%      36.61%
                 vanilla_123_prefer_hate      59.66%      33.32%
            vanilla_123_prefer_offensive      59.66%      33.32%

If prefer_hate vs prefer_offensive shows minimal difference,
the conflict resolution rule has negligible impact.


## Step 11: Bootstrap Significance Test (Constrained)

In [12]:
y_true_bin = convert_to_binary(test_labels)

print(f'{"BOOTSTRAP SIGNIFICANCE TEST (Constrained: RSQwen vs Vanilla)":^70}')
print(f'{"="*70}')

for seed in SEEDS:
    rs_key = f'rsqwen_{seed}'
    van_key = f'vanilla_{seed}'

    rs_f1 = all_results[seed]['constrained'][rs_key]['f1_micro']
    van_f1 = all_results[seed]['constrained'][van_key]['f1_micro']
    print(f'\n  Seed {seed}: RSQwen={rs_f1:.4f}  Vanilla={van_f1:.4f}  Diff={rs_f1-van_f1:+.4f}')

print('\nNote: For full bootstrap test with prediction-level resampling,')
print('save constrained predictions and run R3 analysis notebook.')

     BOOTSTRAP SIGNIFICANCE TEST (Constrained: RSQwen vs Vanilla)     

  Seed 42: RSQwen=0.1525  Vanilla=0.1608  Diff=-0.0084

  Seed 123: RSQwen=0.2059  Vanilla=0.2695  Diff=-0.0636

Note: For full bootstrap test with prediction-level resampling,
save constrained predictions and run R3 analysis notebook.


## Step 12: Save Final Results

In [13]:
for seed in SEEDS:
    print(f'\n  Saving seed {seed}...')

    # Results (free-text + constrained metrics)
    results_path = OUTPUT_DIR / f'results_constrained_seed{seed}.json'
    with open(results_path, 'w') as f:
        json.dump(all_results[seed], f, indent=2, ensure_ascii=False)
    print(f'    Saved: {results_path.name}')

    # Raw outputs (for A3 re-parsing)
    raw_path = OUTPUT_DIR / f'raw_outputs_seed{seed}.json'
    with open(raw_path, 'w') as f:
        json.dump(all_raw_outputs[seed], f, indent=2, ensure_ascii=False)
    print(f'    Saved: {raw_path.name}')

    # Label scores (per-label log-probs)
    scores_path = OUTPUT_DIR / f'label_scores_seed{seed}.json'
    with open(scores_path, 'w') as f:
        json.dump(all_label_scores[seed], f, indent=2, ensure_ascii=False)
    print(f'    Saved: {scores_path.name}')

    # A3 ablation results
    a3_path = OUTPUT_DIR / f'a3_postprocess_ablation_seed{seed}.json'
    with open(a3_path, 'w') as f:
        json.dump(all_a3_results[seed], f, indent=2, ensure_ascii=False)
    print(f'    Saved: {a3_path.name}')

    # Cleanup per-variant checkpoints (no longer needed)
    for variant in ['vanilla', 'rsqwen']:
        cp = OUTPUT_DIR / f'{variant}_{seed}_checkpoint.json'
        if cp.exists():
            cp.unlink()
            print(f'    Cleaned up: {cp.name}')

print(f'\nDONE! All results in: {OUTPUT_DIR}')
print('Files per seed:')
print('  results_constrained_seed{N}.json  - main metrics')
print('  raw_outputs_seed{N}.json          - raw model text')
print('  label_scores_seed{N}.json         - per-label log-probs')
print('  a3_postprocess_ablation_seed{N}.json - ablation results')


  Saving seed 42...
    Saved: results_constrained_seed42.json
    Saved: raw_outputs_seed42.json
    Saved: label_scores_seed42.json
    Saved: a3_postprocess_ablation_seed42.json
    Cleaned up: vanilla_42_checkpoint.json
    Cleaned up: rsqwen_42_checkpoint.json

  Saving seed 123...
    Saved: results_constrained_seed123.json
    Saved: raw_outputs_seed123.json
    Saved: label_scores_seed123.json
    Saved: a3_postprocess_ablation_seed123.json
    Cleaned up: vanilla_123_checkpoint.json
    Cleaned up: rsqwen_123_checkpoint.json

DONE! All results in: /content/drive/MyDrive/ViTHSD/outputs/R4_constrained
Files per seed:
  results_constrained_seed{N}.json  - main metrics
  raw_outputs_seed{N}.json          - raw model text
  label_scores_seed{N}.json         - per-label log-probs
  a3_postprocess_ablation_seed{N}.json - ablation results
